In [1]:
# --------------------------------------------------------
# 
# Postprocess results 
#
# --------------------------------------------------------

In [2]:
# --- Decide which CDR calculations to perform
cdr_calc_list = ["co2_flx",       
                 "camg_flx",
                 "totcat_flx",
                 # "carbalk_flx", # (turn this back on when the runs are re-run with fixed cflx file)
                 "rockdiss"]
# ---

In [ ]:
import fnmatch
import glob
import os
import pickle
import sys
import re
from typing import Tuple

import cmocean.cm as cmo
import fsspec         # for AWS integration
import s3fs
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D  # for custom legend entries (needed for contour plot)
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# --- read in cdr calculation functions  
from ew_workflows import cdr_fxns_postproc as cfp
# ---
#
# TODO: Update emissions calculator to handle time-varying rock application 
# 
#  ----------------------------------------------------------------
# [ RUN NAMES ]
# exe_central1p5 ; exe_central2p0 ; exe_limBio1p5 ; exe_lowCCS1p5 ; 
# exe_lowCDR1p5 ; exe_noBiochar1p5 ; exe_noWeathering1p5 ; 
# exe_ref1p5 ; exe_techBehaviorChange1p5 ; exe_techBehaviorChange2p0 ;
# [ FOCUS RUNS ]
# exe_central1p5 ; exe_techBehaviorChange1p5 ; exe_noWeathering1p5 ; exe_lowCDR1p5
# ----------------------------------------------------------------
#
#
# --- add crop area options (it will save multiple identical clim dirs,
#     but rock-application-specific files will be added later)
repeat_clim_for_crop_area = False
crop_area_frac_arr = np.round(np.linspace(0.6, 1, 5), 2)    # crop_area_frac_arr = np.round(np.linspace(0.1, 1, 10), 2) # from 10% to 100% in 10% increments
# --- add fixed app rate options 
repeat_clim_for_apprate_fixed = True
app_rates_arr = np.array([0.05, 0.1, 0.25, 0.5, 0.75, 1, 2.5, 5, 7.5, 10, 15, 20, 30, 40, 50])
if repeat_clim_for_crop_area and repeat_clim_for_apprate_fixed:
    print("Warning: Must select either crop area or constant apprate to repeat across for how this script is structured!")
# --- add DGSA flag
dgsa_setup = False
# -------------------------------------------------------------------
#
#
# ************************************************
# --- [ settings ]
SAVE_GROUP = "gcam"
EXP_GROUP =  "gcam_opr_full2_cec4" # "gcam_dgsa_v1.2"
EXP_SET = 'exe_central1p5'
dustsp = "gbas2" # gbas # TODO: update `emissions_calculator_df` to use the `dustsp` in the df if mineral is None
multiyear = False
runtype = "yearly_ltm"
spinversion = "v4_repcells1"
extra_tag = spinversion
batchname = f"{EXP_GROUP}_{EXP_SET}_{dustsp}_{runtype}_{extra_tag}.csv"
runtype_labfield = "field"
time_horizon = 80.   # set the time_horizon for computing the integrated flux 
ctrl_conditions = {"dustrate": 0} 
if dgsa_setup: # [ if DGSA -- COLLAPSE TO EACH PARAMETER SET ]
    collapse_cols = ["dustrad", "cec_factor", "apprate_base", "qrun_factor"]
else:
    collapse_cols = None #[ # [ list or None ] # params that are analyzed one at a time, rather than all combinations (we don't want an xr dimension for each param)
#     "dustrad", "secondary_min_rule", "poro_updated",
# ] 
if repeat_clim_for_crop_area:
    dfin_cols_to_keep = [   # become dimensions in later xr dataset
        "site", "crop_fraction" # (need dustrad for emissions calc)
    ] 
elif repeat_clim_for_apprate_fixed:
    dfin_cols_to_keep = [   # become dimensions in later xr dataset
        "site", "apprate_base", "dustrad", "cec_factor"   # for "*_psdfull_3axis1"
    ]     
elif dgsa_setup:
    dfin_cols_to_keep = [   # become dimensions in later xr dataset
        "site", "dustrad", "cec_factor", "apprate_base", "qrun_factor"
    ]
dfin_cols_to_keep_time = dfin_cols_to_keep + ["time"]
save_results = True
savepath = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/{SAVE_GROUP}"
# ************************************************


# --- read in the batch .csv
batchdir = "s3://carbonplan-carbon-removal/ew-workflows-data/scepter/batch/"
# update to the "postproc" batch csv (merges asynch case + ctrls) if available
_fs, _pp = s3fs.S3FileSystem(), batchname.replace(".csv", "_postproc.csv")
if _fs.exists(batchdir + _pp):
    if _fs.info(batchdir + _pp)["LastModified"] < _fs.info(batchdir + batchname)["LastModified"]:
        raise RuntimeError(f"{_pp} is stale -- rerun scratch/make_postproc_csvs.py")
    batchname = _pp
print(f"batch csv: {batchname}")
# read in
dfin = pd.read_csv(os.path.join(batchdir, batchname))

# --- derive ctrl_params based on what actually varies in this batch
_cec_varies = "cec_factor" in dfin.columns and dfin["cec_factor"].nunique(dropna=True) > 1
ctrl_params = ["site", "cec_factor"] if _cec_varies else None
# ctrl_params = ["site"]

# ---
# groundwork
outdir = f"s3://carbonplan-carbon-removal/SCEPTER/scepter_output/{EXP_GROUP}/{EXP_SET}"

batch csv: gcam_dgsa_v1.2_exe_central1p5_gbas_yearly_ltm_v4_repcells1_postproc.csv


In [10]:
# --- add crop area column if asked (stripped from the aws path)
if repeat_clim_for_crop_area & ("crop_fraction" in dfin_cols_to_keep):
    dfin['crop_fraction'] = dfin['aws_bucket'].str.split('/').str[-1]
# --- add column for the full run ID
if multiyear:
    dfin["newrun_id_full"] = dfin["newrun_id"] + f"_composite_{runtype_labfield}"
else:
    dfin['newrun_id_full'] = dfin['newrun_id'] + "_" + dfin['dustsp'] + "_" + runtype_labfield + "_tau"+dfin["duration"].astype(float).astype(str).str.replace(".", "p")  # (duration has to be turned into float first because otherwise we miss the decimal pt)
# prepend aws_bucket subdir if runs are not all in the same, single outdir (need for proper path building bc we hardcoded a pass to a single outdir later)
if repeat_clim_for_crop_area & ("crop_fraction" in dfin_cols_to_keep):
    bucket_base = outdir.rstrip('/')
    bucket_suffix = dfin['aws_bucket'].str.rstrip('/').apply(
        lambda b: b[len(bucket_base) + 1:] if b.startswith(bucket_base + '/') else ''
    )
    mask = bucket_suffix != ''
    dfin.loc[mask, 'newrun_id_full'] = bucket_suffix[mask] + '/' + dfin.loc[mask, 'newrun_id_full']
if repeat_clim_for_apprate_fixed & ("apprate_base" in dfin_cols_to_keep):
    bucket_base = outdir.rstrip('/')
    bucket_suffix = dfin['aws_bucket'].str.rstrip('/').apply(
        lambda b: b[len(bucket_base) + 1:] if b.startswith(bucket_base + '/') else ''
    )
    mask = bucket_suffix != ''
    dfin.loc[mask, 'newrun_id_full'] = bucket_suffix[mask] + '/' + dfin.loc[mask, 'newrun_id_full']
elif dgsa_setup & ("apprate_base" in dfin_cols_to_keep):
    bucket_base = outdir.rstrip('/')
    bucket_suffix = dfin['aws_bucket'].str.rstrip('/').apply(
        lambda b: b[len(bucket_base) + 1:] if b.startswith(bucket_base + '/') else ''
    )
    mask = bucket_suffix != ''
    dfin.loc[mask, 'newrun_id_full'] = bucket_suffix[mask] + '/' + dfin.loc[mask, 'newrun_id_full']
# identify the control run(s)
dfin = cfp.find_control_runs(dfin, ctrl_conditions)
# add a column for the dustrate in ton_ha_yr
if "dustrate" in dfin.columns:
    dfin["dustrate_ton_ha_yr"] = dfin["dustrate"] / 100 

# add dustrad if we only have psdrain_meanRad
if "dustrad" not in dfin.columns:
    dfin['dustrad'] = dfin['psdrain_meanRad'] * 1e6 # [ convert m to um ]

# --- extract sites
sitenames = dfin['site'].unique()

# --- create the collapsed col
# `dfin_cols_to_keep` / `dfin_cols_to_keep_time` stay minimal (site + collapsed_inputs)
# since they become xr dims later in `cdr_ds` -- keeping the individual collapse_cols
# in there too would make `cdr_ds` build a dim per param ON TOP OF collapsed_inputs,
# i.e. a mostly-NaN site x dustrad x cec_factor x apprate_base x qrun_factor x
# collapsed_inputs cross product (this is what was blowing up memory).
# `dfin_cols_to_keep_flx` / `_flx_time` are the versions passed to read_postproc_flux /
# read_cation_budget_flx / cdr_int_per_group -- they keep the individual collapse_cols
# as plain (non-indexed) columns so that (a) ctrl_params matching (e.g. on cec_factor)
# still works, and (b) those numeric values ride through automatically as extra data
# variables once cdr_ds indexes only on the minimal dims above (any dataframe column
# not named in `dims` passes through `set_index(dims)` as a plain variable, not a
# new dimension). When collapse_cols is unused, these are just aliases of the
# originals, so downstream cells can reference `_flx` unconditionally.
dfin_cols_to_keep_flx = dfin_cols_to_keep
dfin_cols_to_keep_flx_time = dfin_cols_to_keep_time
if collapse_cols:
    dfin["collapsed_inputs"] = dfin[collapse_cols].apply(
       lambda row: " | ".join(f"{col}={row[col]}" for col in collapse_cols),
        axis=1
    )
    # update the ds coords
    if dgsa_setup:
        dfin_cols_to_keep_flx = dfin_cols_to_keep + ["collapsed_inputs"]
        dfin_cols_to_keep_flx_time = dfin_cols_to_keep_time + ["collapsed_inputs"]
        dfin_cols_to_keep = list(set(dfin_cols_to_keep) - set(collapse_cols)) + ["collapsed_inputs"]
        dfin_cols_to_keep_time = list(set(dfin_cols_to_keep_time) - set(collapse_cols)) + ["collapsed_inputs"]
    else:
        dfin_cols_to_keep = dfin_cols_to_keep + ["collapsed_inputs"]
        dfin_cols_to_keep_time = dfin_cols_to_keep_time + ["collapsed_inputs"]
        dfin_cols_to_keep_flx = dfin_cols_to_keep
        dfin_cols_to_keep_flx_time = dfin_cols_to_keep_time

In [11]:
# # --- [ QUICK TEST: Shorten dfin ] ------------------------------------------------------
# # --- restrict to 3 sites (including control runs) to keep this test fast
# test_sites = ['USA'] #.. 'China', 'India']
# dfin  = dfin[dfin['site'].isin(test_sites)].reset_index(drop=True)
# print(f"Test dfin: {len(dfin)} rows  "
#       f"({dfin['ctrl_run'].sum()} controls, "
#       f"{(~dfin['ctrl_run']).sum()} cases)")
# dfin[['site', 'apprate_base', 'cec_factor', 'ctrl_run']].drop_duplicates().head(10)
# # ---------------------------------------------------------------------------------------

# Flux postprocessing

In [12]:
# --- read in the postprocessed flux data 
# [ integrated flux ]   
flx_dict_int = cfp.read_postproc_flux(
    dfin, outdir, cdr_calc_list, dfin_cols_to_keep_flx, 
    flx_type="int_flx", rockdiss_feedstock=None
)
# [ flux ]
flx_dict = cfp.read_postproc_flux(
    dfin, outdir, cdr_calc_list, dfin_cols_to_keep_flx, 
    flx_type="flx", rockdiss_feedstock=None
)

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


## Optional: Cation budget
Take feedstock as the cation source and track all non-negligible sinks. 

In [13]:
# --- run the function
# [ setup ]
newflxname = "catbudget_flx"
cdr_calc_list += [newflxname]
# [ run ]
print(f"solving {newflxname}")
dfcat_int, dfcat_trans, df_skipped = cfp.read_cation_budget_flx(  # compute both time-int and transient results
    dfin,
    outdir,
    dfin_cols_to_keep_flx,
    verbose=False,
    return_skipped=True
)
# add into flx_dicts
flx_dict[newflxname] = dfcat_trans
flx_dict_int[newflxname] = dfcat_int
# ---
# print("\nIntegrated shape:", dfcat_int.shape)
# print(dfcat_int.columns.tolist())
# dfcat_int.head(3)

solving catbudget_flx


In [14]:
# ——————————————————————————
# Do we need to rerun anything for the catbudget? (i.e., was anything skipped?)
df_skipped

,run_index,runname,missing_cations,n_missing,reason,site,dustrad,cec_factor,apprate_base,qrun_factor,collapsed_inputs


## CDR fluxes

In [15]:
# --- compute CDR fluxes (with losses considered)
# ---
# [ time-integrated ]
cdr_dict_int_full, cdr_dict_int_sum = cfp.cdr_int_per_group(
    flx_dict_int, time_horizon, 
    cdr_calc_list, dfin_cols_to_keep_flx,
    bysite = True,
    ctrl_params = ctrl_params,
)
# [ transient ]
cdr_dict_full, cdr_dict_sum = cfp.cdr_int_per_group(
    flx_dict, time_horizon, 
    cdr_calc_list, dfin_cols_to_keep_flx,
    bysite = True,
    ctrl_params = ctrl_params,
)

# **************************************************
if save_results:
    # create a dictionary of ds's to save 
    df_dict_save = {
        "cdr_dfs_int_sum.pkl": cdr_dict_int_sum,
        "cdr_dfs_int.pkl": cdr_dict_int_full,
        "cdr_dfs_sum.pkl": cdr_dict_sum,
        "cdr_dfs_full.pkl": cdr_dict_full,
    }
# **************************************************

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving catbudget_flx
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving catbudget_flx


In [16]:
# --- compute CDR w/ loss rates and convert to xr datasets
outds_int = cfp.cdr_ds(
    cdr_dict = cdr_dict_int_sum, dims = dfin_cols_to_keep, 
    cdr_calc_list = cdr_calc_list, skip_loss=False, convert_time_to_timestep = False,
)
# --- repeat for time-dependent time-integrated data
outds_int_time = cfp.cdr_ds(
    cdr_dict = cdr_dict_int_full, dims = dfin_cols_to_keep_time, 
    cdr_calc_list = cdr_calc_list, skip_loss = True,
    convert_time_to_timestep = True,
)
# --- repeat for time-transient data (note dims now includes "time")
outds = cfp.cdr_ds(
    cdr_dict = cdr_dict_full, dims = dfin_cols_to_keep_time, 
    cdr_calc_list = cdr_calc_list, skip_loss = True, 
    convert_time_to_timestep = True,
)

# --- clean up the passthrough sensitivity-parameter columns (dustrad, cec_factor,
# apprate_base, qrun_factor). They ride into these datasets as incidental leftover
# columns from each individual calc (co2_flx, camg_flx, rockdiss, ...) rather than
# through any explicit dtype-aware path, so they can (a) come out as "object" dtype
# once xr.merge reconciles the 5 separately-built datasets, (b) inherit bogus
# units/cdr_fxn attrs from whichever calc's `*_cdr_ds` helper loop tagged them last
# (that loop in the shared library blindly tags ALL data_vars, not just its own
# flux columns), and (c) pick up a spurious "time" dimension in the time-resolved
# datasets -- these params are constant per (site, collapsed_inputs) case, but the
# underlying long dataframe repeats the same value on every timestep row, so
# from_dataframe reshapes them into a full (site, collapsed_inputs, time) variable.
# Fix all three here rather than in the shared ew_workflows library.
if collapse_cols:
    def _clean_passthrough_params(ds):
        for col in collapse_cols:
            if col in ds.variables:
                da = ds[col].astype(float)
                if "time" in da.dims:
                    da = da.isel(time=0, drop=True)  # constant across time -- drop the redundant dim
                da.attrs = {}
                ds[col] = da
        return ds

    outds_int = _clean_passthrough_params(outds_int)
    outds_int_time = _clean_passthrough_params(outds_int_time)
    outds = _clean_passthrough_params(outds)

# **************************************************
if save_results:
    # create a dictionary of ds's to save 
    ds_dict = {
        "ds_cdr_time-integrated.zarr": outds_int,
        "ds_cdr.zarr": outds_int_time,
        "ds_cdr_transient.zarr": outds, 
    }
# **************************************************


solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving catbudget_flx
solving co2_flx
solving camg_flx


/Users/tylerkukla/Github/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1706: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout_dict.values())
/Users/tylerkukla/Github/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1706: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout_dict.values())
/Use

solving totcat_flx
solving rockdiss
solving catbudget_flx


/Users/tylerkukla/Github/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1706: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout_dict.values())
/Users/tylerkukla/Github/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1706: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout_dict.values())
/Use

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving catbudget_flx


/Users/tylerkukla/Github/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1706: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout_dict.values())
/Users/tylerkukla/Github/ew-workflows/src/ew_workflows/cdr_fxns_postproc.py:1706: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dsout = xr.merge(dsout_dict.values())
/Use

### Optional: Add time-transient emissions

In [17]:
# # --- get emissions (over time if expand_over_time=True)
# # set the inputs ***************
# p80_input = 1.3e3
# Efactor_org = "MRO"
# # ******************************
# ds_emiss_int_time = cfp.emissions_calculator_ds(
#     outds_int_time,
#     dustsp,
#     time_horizon,
#     p80_input,
#     Efactor_org,
#     expand_over_time=True,
# )
# # **************************************************
# if save_results:
#     # create a dictionary of ds's to save 
#     ds_dict['ds_emiss.zarr'] = ds_emiss_int_time
# # **************************************************


In [19]:
# --- save results **************************************************************************
# 
if save_results:
    if ds_dict.get("ds_emiss.zarr"):
        # create the dictionary we'll save as the resource file
        savedict = cfp.create_save_dict_oneFS_emiss(
            EXP_SET, cdr_calc_list, dfin_cols_to_keep, collapse_cols, 
            batchname, multiyear, time_horizon, outdir, batchdir, p80_input,
            Efactor_org, EXP_GROUP=EXP_GROUP, 
        )
    else:
        # create the dictionary we'll save as the resource file
        savedict = cfp.create_save_dict_oneFS_noEmiss(
            EXP_SET, cdr_calc_list, dfin_cols_to_keep, collapse_cols, 
            batchname, multiyear, time_horizon, outdir, batchdir, 
        )
    
    # write the files
    fname_res = "_calc_inputs.res"  
    base_name = f"{EXP_GROUP}/{EXP_SET}"
    saved_here = cfp.save_batch_postproc_oneSet(
        savedict, savepath, base_name, fname_res, df_dict_save,
        ds_dict, save_directory=None, # (None means we make a new directory)
    )
# *********************************************************************************************    


Directory created: s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/gcam/gcam_dgsa_v1.2/exe_central1p5_001


# Profile postprocessing

In [ ]:
# --- prepare the post-processing inputs
# 
batch_axes = dfin_cols_to_keep
if "dustrate_ton_ha_yr" not in batch_axes: # make sure we distinguish between case / ctrl
    batch_axes.append("dustrate_ton_ha_yr")
#
# dictionary for which files to process in the batch profile functions
proc_dict = {
    "adsorbed": True,
    "adsorbed_percCEC": True,
    "adsorbed_ppm": True,
    "aqueous": True,
    "aqueous_total": False,
    "bulksoil": True,
    "exchange_total": True,
    "gas": True,
    "rate": True,
    "soil_ph": True,
    "solid": True,
    "solid_sp_saturation": True, 
    "solid_volumePercent": True,
    "solid_weightPercent": True,
    "specific_surface_area": True,
    "surface_area": True,
}

# saved_here = "s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/gcam/gcam_opr_psdfull_3axis2/exe_central1p5_001"


In [ ]:
# --- compile all the profile data into a single dataset
# save_path=saved_here writes each variable to zarr immediately and frees it,
# so only one variable's data is in memory at a time.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning) # ... kicking a can down the road

zarr_paths = cfp.prof_batchprocess_allvars(
    outdir, dustsp, dfin, batch_axes, proc_dict,
    print_loop_updates=False,
    save_path=saved_here,
)
# zarr_paths is a dict mapping variable name -> zarr store path;
# read any variable back with: xr.open_zarr(zarr_paths['solid'])

### Save profile results

In [ ]:
# --- profile data is already saved to zarr stores in `saved_here`
#     (done inline above via save_path=saved_here in prof_batchprocess_allvars).
# To use the old approach (hold all vars in memory, then save at the end),
# remove save_path from the cell above and uncomment this cell:
#
# cfp.save_batch_postproc_profOnly(
#     dsdict, dustsp, save_directory=saved_here,
#     base_path=None, base_dir_name=None, use_zarr=True,
# )

In [ ]:
# ---
# import sys
# print(sys.executable)
# print(sys.version)